# 04.1 Train The Student With TRL/PEFT

This is the PyTorch/Hugging Face comparison run for notebook 04. It uses the same teacher SFT rows, the same max sequence length, the same LoRA rank/scale, the same learning rate, and the same train/validation split logic.

One important difference: MLX-LM trains the MLX checkpoint `mlx-community/Qwen3.5-0.8B-MLX-bf16`; TRL/PEFT trains a PyTorch Hugging Face checkpoint, so this notebook uses `Qwen/Qwen3.5-0.8B`. The tokenizer family and data format stay aligned, but the runtime is different. That is exactly the speed comparison we want.

Do not run this while another long training job is already using memory.

In [ ]:
from __future__ import annotations

from statistics import mean
import json
import math
import os
from pathlib import Path
import sys

cwd = Path.cwd()
ROOT = cwd if (cwd / "common" / "config.py").exists() else cwd.parent
if not (ROOT / "common" / "config.py").exists():
    raise RuntimeError("Run this notebook from the repo root or a blog folder under the repo root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from common import config as cfg
from common import sft_rows

paths = cfg.setup_notebook_paths(blog_dir_name="1-distilling-a-0-8b-tool-calling-agent")
ROOT = paths.root
BLOG_DIR = paths.blog_dir
DATA_DIR = paths.data_dir
OUTPUT_DIR = paths.output_dir
ENV_PATH = paths.env_path
DOTENV_LOADED = paths.dotenv_loaded

TAU_BENCH_USER_SIMULATOR_LLM = cfg.required_env("TAU_BENCH_USER_SIMULATOR_LLM")
USER_SIMULATOR_SLUG = cfg.filename_slug(TAU_BENCH_USER_SIMULATOR_LLM)
TEACHER_MODEL = cfg.TEACHER_MODEL
TEACHER_PROVIDER = "vllm_raw"
TEACHER_SLUG = cfg.filename_slug(TEACHER_MODEL)

STUDENT_MODEL = "Qwen/Qwen3.5-0.8B"
STUDENT_MLX_REFERENCE_MODEL = cfg.STUDENT_MODEL
STUDENT_SLUG = cfg.filename_slug(STUDENT_MODEL)

SFT_WORK_DIR = OUTPUT_DIR / f"{STUDENT_SLUG}_tau3_retail_sft_trl_peft"
HF_DATA_DIR = SFT_WORK_DIR / "hf_tokenized_data"
ADAPTER_OUTPUT_DIR = SFT_WORK_DIR / f"{STUDENT_SLUG}_tau3_retail_trl_lora_adapter"
SFT_SOURCE_PATH = OUTPUT_DIR / f"{TEACHER_SLUG}_{TEACHER_PROVIDER}_tau3_bench_retail_train_successful_sft_chat_rows_{USER_SIMULATOR_SLUG}.jsonl"

print("Project root:", ROOT)
print("Loaded .env:", DOTENV_LOADED, "from", ENV_PATH)
print("SFT source path:", SFT_SOURCE_PATH)
print("TRL/PyTorch student model:", STUDENT_MODEL)
print("MLX comparison student model:", STUDENT_MLX_REFERENCE_MODEL)
print("Work dir:", SFT_WORK_DIR)

## Dependency Check

This notebook depends on TRL and PEFT. They are project dependencies now, but your local environment still needs to be synced before running this notebook.

In [ ]:
try:
    import peft
    import trl
except ModuleNotFoundError as error:
    raise RuntimeError(
        "Missing TRL/PEFT runtime packages. After the current training job is done, run `uv sync` from the repo root."
    ) from error

print("TRL version:", getattr(trl, "__version__", "unknown"))
print("PEFT version:", getattr(peft, "__version__", "unknown"))

## Load Teacher Rows

Each row is one teacher action target. The input is the conversation history up to that point, plus the tool schemas. The label is only the next assistant message, not the prompt.

In [ ]:
if not SFT_SOURCE_PATH.exists():
    candidates = sorted(OUTPUT_DIR.glob("*_tau3_bench_retail_train_successful_sft_chat_rows_*.jsonl"))
    print("Could not find the expected SFT source path.")
    print("Expected:", SFT_SOURCE_PATH)
    print("Candidate SFT files:")
    for candidate in candidates:
        print(" -", candidate)
    raise FileNotFoundError(SFT_SOURCE_PATH)

rows = cfg.load_jsonl(SFT_SOURCE_PATH)
if not rows:
    raise RuntimeError("The SFT source file exists but contains no rows.")

print("Loaded SFT rows:", len(rows))
print("Unique task ids:", len({row["task_id"] for row in rows}))
print("Tool-call target rows:", sum(row.get("is_tool_call_target", False) for row in rows))
print("Natural-language target rows:", sum(not row.get("is_tool_call_target", False) for row in rows))
print("First row roles:", [message["role"] for message in rows[0]["messages"][-5:]])
print()
print("First target preview:")
print(rows[0]["target_text"][:800])

## Measure Sequence Lengths

We measure with the same Qwen chat template before filtering. Long rows are dropped rather than silently truncated because truncating the middle of an agent trajectory would teach the model from broken context.

In [ ]:
tokenizer = cfg.load_tokenizer(STUDENT_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.model_max_length = max(int(getattr(tokenizer, "model_max_length", 0) or 0), 16500)

length_rows = []
for index, row in enumerate(rows):
    token_count = sft_rows.mlx_chat_row_token_length(row, tokenizer)
    length_rows.append({
        "index": index,
        "task_id": row["task_id"],
        "source_message_index": row["source_message_index"],
        "tokens": token_count,
        "is_tool_call_target": row.get("is_tool_call_target", False),
    })

lengths = [item["tokens"] for item in length_rows]
length_stats = {
    "min": min(lengths),
    "p50": cfg.percentile_int(lengths, 0.50),
    "p90": cfg.percentile_int(lengths, 0.90),
    "p95": cfg.percentile_int(lengths, 0.95),
    "max": max(lengths),
    "mean": round(mean(lengths), 1),
}
print(json.dumps(length_stats, indent=2))
print()
print("Longest rows:")
for item in sorted(length_rows, key=lambda item: item["tokens"], reverse=True)[:10]:
    print(item)

## Choose The Matching TRL/PEFT Config

These values mirror notebook 04 as closely as the runtimes allow. `LORA_NUM_LAYERS = 1` means we adapt only the final transformer block. In PEFT terms, we target the attention projections and MLP projections inside that final block.

In [ ]:
from transformers import AutoConfig

MAX_SEQ_LENGTH = 16500
LORA_NUM_LAYERS = 1
LORA_RANK = 8
LORA_SCALE = 20.0
LEARNING_RATE = 1e-5
BATCH_SIZE = 4
ITERS = -1  # -1 means one pass over the kept rows, measured in batches.
GRAD_ACCUMULATION_STEPS = 1
VALIDATION_TASK_FRACTION = 0.10
SPLIT_SEED = 42
TRAINING_SEED = 0
GRAD_CHECKPOINT = True
RESUME_FROM_CHECKPOINT = None  # Set to "latest" to resume optimizer state from the latest TRL checkpoint.

model_config = AutoConfig.from_pretrained(STUDENT_MODEL, trust_remote_code=True)
MODEL_NUM_LAYERS = int(model_config.num_hidden_layers)
LORA_LAYER_INDICES = list(range(MODEL_NUM_LAYERS - LORA_NUM_LAYERS, MODEL_NUM_LAYERS))
LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

rows_that_fit = [row for row, meta in zip(rows, length_rows) if meta["tokens"] <= MAX_SEQ_LENGTH]
rows_too_long = [meta for meta in length_rows if meta["tokens"] > MAX_SEQ_LENGTH]

print("Student HF model:", STUDENT_MODEL)
print("Max sequence length:", MAX_SEQ_LENGTH)
print("Model layers:", MODEL_NUM_LAYERS)
print("LoRA layer indices:", LORA_LAYER_INDICES)
print("LoRA target modules:", LORA_TARGET_MODULES)
print("LoRA rank:", LORA_RANK)
print("LoRA scale / PEFT alpha:", LORA_SCALE)
print("Learning rate:", LEARNING_RATE)
print("Batch size:", BATCH_SIZE)
print("Gradient accumulation steps:", GRAD_ACCUMULATION_STEPS)
print("Gradient checkpointing:", GRAD_CHECKPOINT)
print("Training seed:", TRAINING_SEED)
print("Resume checkpoint:", RESUME_FROM_CHECKPOINT)
print("Rows kept:", len(rows_that_fit))
print("Rows longer than max sequence length:", len(rows_too_long))

if rows_too_long:
    print()
    print("Longest dropped rows:")
    for item in sorted(rows_too_long, key=lambda item: item["tokens"], reverse=True)[:10]:
        print(item)

if not rows_that_fit:
    raise RuntimeError("No rows fit the current MAX_SEQ_LENGTH. Increase MAX_SEQ_LENGTH or collect shorter rows.")

## Build Prompt-Masked TRL Datasets

TRL can consume a pre-tokenized `datasets.Dataset`. We use that path so the labels are exact: prompt tokens are `-100`, and only the teacher's next assistant message contributes loss.

In [ ]:
from datasets import Dataset

train_rows, validation_rows, validation_task_ids = sft_rows.split_sft_rows_by_task_id(
    rows_that_fit,
    validation_task_fraction=VALIDATION_TASK_FRACTION,
    seed=SPLIT_SEED,
)
if not validation_rows:
    validation_rows = train_rows[:1]


def tokenize_sft_row(row: dict, row_index: int) -> dict | None:
    target_message = row["messages"][-1]
    if target_message["role"] != "assistant":
        raise ValueError(f"SFT row {row_index} target is not an assistant message.")

    prompt_text = tokenizer.apply_chat_template(
        row["messages"][:-1],
        tools=row.get("tools"),
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    target_ids = tokenizer.encode(target_message["content"], add_special_tokens=False)
    if tokenizer.eos_token_id is not None and (not target_ids or target_ids[-1] != tokenizer.eos_token_id):
        target_ids.append(tokenizer.eos_token_id)

    input_ids = prompt_ids + target_ids
    if len(input_ids) > MAX_SEQ_LENGTH:
        return None

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": [-100] * len(prompt_ids) + target_ids,
    }

train_examples = []
for row_index, row in enumerate(train_rows):
    example = tokenize_sft_row(row, row_index)
    if example is not None:
        train_examples.append(example)

validation_examples = []
for row_index, row in enumerate(validation_rows):
    example = tokenize_sft_row(row, row_index)
    if example is not None:
        validation_examples.append(example)

if not train_examples:
    raise RuntimeError("No tokenized train examples fit MAX_SEQ_LENGTH.")
if not validation_examples:
    validation_examples = train_examples[:1]

train_dataset = Dataset.from_list(train_examples)
validation_dataset = Dataset.from_list(validation_examples)
train_token_lengths = [len(example["input_ids"]) for example in train_examples]
validation_token_lengths = [len(example["input_ids"]) for example in validation_examples]

HF_DATA_DIR.mkdir(parents=True, exist_ok=True)
train_dataset.save_to_disk(HF_DATA_DIR / "train")
validation_dataset.save_to_disk(HF_DATA_DIR / "valid")

if ITERS <= 0:
    ITERS = max(1, math.ceil(len(train_dataset) / max(1, BATCH_SIZE * GRAD_ACCUMULATION_STEPS)))

CHECKPOINT_EVERY = max(1, min(25, ITERS))
STEPS_PER_EVAL = max(1, min(25, ITERS))
STEPS_PER_REPORT = 1

print("Train rows before tokenization:", len(train_rows))
print("Validation rows before tokenization:", len(validation_rows))
print("Train examples:", len(train_dataset))
print("Validation examples:", len(validation_dataset))
print("Train max tokens:", max(train_token_lengths))
print("Validation max tokens:", max(validation_token_lengths))
print("Validation task ids:", sorted(validation_task_ids))
print("Iters / max_steps:", ITERS)
print("Checkpoint every:", CHECKPOINT_EVERY)
print("Eval every:", STEPS_PER_EVAL)
print("HF tokenized data dir:", HF_DATA_DIR)

## Train With TRL SFTTrainer And PEFT LoRA

This is the cell that actually trains. It saves TRL checkpoints, including optimizer state, under `ADAPTER_OUTPUT_DIR/checkpoint-*`. If training stops and the batch/config is unchanged, set `RESUME_FROM_CHECKPOINT = "latest"` in the config cell and rerun from there.

In [ ]:
import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM, DataCollatorForSeq2Seq
from transformers.trainer_utils import get_last_checkpoint
from trl import SFTConfig, SFTTrainer

RUN_TRAINING = True
os.environ["HF_HUB_DISABLE_XET"] = "1"

model_dtype = torch.bfloat16 if torch.backends.mps.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL,
    torch_dtype=model_dtype,
    trust_remote_code=True,
)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

peft_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=LORA_RANK,
    lora_alpha=int(LORA_SCALE),
    lora_dropout=0.0,
    target_modules=LORA_TARGET_MODULES,
    layers_to_transform=LORA_LAYER_INDICES,
    layers_pattern="layers",
    bias="none",
)

training_args = SFTConfig(
    output_dir=str(ADAPTER_OUTPUT_DIR),
    max_length=MAX_SEQ_LENGTH,
    max_steps=ITERS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=STEPS_PER_REPORT,
    eval_strategy="steps",
    eval_steps=STEPS_PER_EVAL,
    save_strategy="steps",
    save_steps=CHECKPOINT_EVERY,
    save_total_limit=3,
    bf16=model_dtype == torch.bfloat16,
    fp16=False,
    gradient_checkpointing=GRAD_CHECKPOINT,
    optim="adamw_torch",
    report_to=[],
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    seed=TRAINING_SEED,
    data_seed=TRAINING_SEED,
    packing=False,
    dataset_kwargs={"skip_prepare_dataset": True},
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
    return_tensors="pt",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
    data_collator=data_collator,
)

resume_checkpoint = None
if RESUME_FROM_CHECKPOINT == "latest" and ADAPTER_OUTPUT_DIR.exists():
    resume_checkpoint = get_last_checkpoint(str(ADAPTER_OUTPUT_DIR))
elif RESUME_FROM_CHECKPOINT:
    resume_checkpoint = str(RESUME_FROM_CHECKPOINT)

print("Training config:")
print(json.dumps(cfg.make_json_safe(training_args.to_dict()), indent=2)[:4000])
print("Adapter output dir:", ADAPTER_OUTPUT_DIR)
print("Resume checkpoint:", resume_checkpoint)
print("Model dtype:", model_dtype)
print("Run training:", RUN_TRAINING)

if RUN_TRAINING:
    train_output = trainer.train(resume_from_checkpoint=resume_checkpoint)
    trainer.save_model(str(ADAPTER_OUTPUT_DIR))
    tokenizer.save_pretrained(str(ADAPTER_OUTPUT_DIR))
    training_result = {
        "metrics": train_output.metrics,
        "global_step": trainer.state.global_step,
        "log_history": trainer.state.log_history,
    }
else:
    training_result = {
        "metrics": {},
        "global_step": 0,
        "log_history": [],
    }

TRAINER_LOG_HISTORY_PATH = ADAPTER_OUTPUT_DIR / "trainer_log_history.json"
ADAPTER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAINER_LOG_HISTORY_PATH.write_text(json.dumps(cfg.make_json_safe(training_result["log_history"]), indent=2), encoding="utf-8")

print("Training result:")
print(json.dumps(cfg.make_json_safe(training_result), indent=2)[:4000])
print("Saved trainer log history to:", TRAINER_LOG_HISTORY_PATH)

## Plot Training Loss

Run this after training to compare TRL loss shape and speed with the MLX notebook.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

history_rows = []
if "training_result" in globals() and training_result.get("log_history"):
    history_rows = training_result["log_history"]
elif (ADAPTER_OUTPUT_DIR / "trainer_log_history.json").exists():
    history_rows = json.loads((ADAPTER_OUTPUT_DIR / "trainer_log_history.json").read_text(encoding="utf-8"))

history_df = pd.DataFrame(history_rows)
if history_df.empty:
    print("No TRL trainer history found yet.")
else:
    display(history_df.tail(10))
    if "loss" in history_df:
        train_df = history_df.dropna(subset=["loss"])
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(train_df["step"], train_df["loss"], marker="o", label="train loss")
        if "eval_loss" in history_df:
            eval_df = history_df.dropna(subset=["eval_loss"])
            if not eval_df.empty:
                ax.plot(eval_df["step"], eval_df["eval_loss"], marker="s", label="validation loss")
        ax.set_title("TRL/PEFT LoRA Training Loss")
        ax.set_xlabel("Step")
        ax.set_ylabel("Loss")
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.show()

    runtime_columns = [column for column in ["train_runtime", "train_samples_per_second", "train_steps_per_second"] if column in history_df]
    if runtime_columns:
        print(history_df[runtime_columns].dropna(how="all").tail())

## Save Training Metadata

This gives notebook 05, or any later comparison, a single JSON file describing exactly which runtime/config produced the adapter.

In [ ]:
TRAINING_METADATA_PATH = SFT_WORK_DIR / "training_metadata_trl_peft.json"
training_metadata = {
    "student_base_model": STUDENT_MODEL,
    "student_mlx_reference_model": STUDENT_MLX_REFERENCE_MODEL,
    "teacher_model": TEACHER_MODEL,
    "teacher_provider": TEACHER_PROVIDER,
    "sft_source_path": str(SFT_SOURCE_PATH),
    "hf_data_dir": str(HF_DATA_DIR),
    "adapter_output_dir": str(ADAPTER_OUTPUT_DIR),
    "raw_rows": len(rows),
    "rows_kept": len(rows_that_fit),
    "rows_dropped_over_length": len(rows_too_long),
    "train_examples": len(train_dataset),
    "validation_examples": len(validation_dataset),
    "length_stats": length_stats,
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_num_layers": LORA_NUM_LAYERS,
    "lora_layer_indices": LORA_LAYER_INDICES,
    "lora_target_modules": LORA_TARGET_MODULES,
    "lora_rank": LORA_RANK,
    "lora_scale": LORA_SCALE,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUMULATION_STEPS,
    "iters": ITERS,
    "checkpoint_every": CHECKPOINT_EVERY,
    "steps_per_eval": STEPS_PER_EVAL,
    "steps_per_report": STEPS_PER_REPORT,
    "training_seed": TRAINING_SEED,
    "grad_checkpoint": GRAD_CHECKPOINT,
    "validation_task_fraction": VALIDATION_TASK_FRACTION,
    "split_seed": SPLIT_SEED,
    "resume_from_checkpoint": RESUME_FROM_CHECKPOINT,
    "run_training": RUN_TRAINING if "RUN_TRAINING" in globals() else None,
    "training_result": training_result if "training_result" in globals() else None,
    "adapter_model_exists": (ADAPTER_OUTPUT_DIR / "adapter_model.safetensors").exists(),
    "adapter_config_exists": (ADAPTER_OUTPUT_DIR / "adapter_config.json").exists(),
}
SFT_WORK_DIR.mkdir(parents=True, exist_ok=True)
TRAINING_METADATA_PATH.write_text(json.dumps(cfg.make_json_safe(training_metadata), indent=2), encoding="utf-8")
print(json.dumps(cfg.make_json_safe(training_metadata), indent=2)[:4000])
print("Saved metadata to:", TRAINING_METADATA_PATH)

## Smoke Test Adapter Loading

Run this after training if you want a quick sanity check before notebook 05. It loads the PEFT adapter and generates one target from a validation prompt.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

if not (ADAPTER_OUTPUT_DIR / "adapter_model.safetensors").exists():
    raise FileNotFoundError(ADAPTER_OUTPUT_DIR / "adapter_model.safetensors")

smoke_row = validation_rows[0]
base_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL,
    torch_dtype=torch.bfloat16 if torch.backends.mps.is_available() else torch.float32,
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, str(ADAPTER_OUTPUT_DIR))
model.eval()
if torch.backends.mps.is_available():
    model.to("mps")

smoke_prompt = tokenizer.apply_chat_template(
    smoke_row["messages"][:-1],
    tools=smoke_row.get("tools"),
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
inputs = tokenizer(smoke_prompt, return_tensors="pt")
if torch.backends.mps.is_available():
    inputs = {key: value.to("mps") for key, value in inputs.items()}

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

new_tokens = output_ids[0, inputs["input_ids"].shape[-1]:]
smoke_text = tokenizer.decode(new_tokens, skip_special_tokens=False)
print("Expected teacher target:")
print(smoke_row["target_text"][:1000])
print()
print("Student with TRL adapter generated:")
print(smoke_text)

## What This Notebook Produces

After a successful run, the important artifacts are:

- tokenized train/validation datasets under `HF_DATA_DIR`
- TRL checkpoints and the final PEFT adapter under `ADAPTER_OUTPUT_DIR`
- `training_metadata_trl_peft.json` for comparing config and speed against the MLX run
- `trainer_log_history.json` for plotting loss and runtime statistics